# Data Analysis Notebook

## Objective
Perform exploratory data analysis (EDA) on the cleaned food price inflation dataset to identify patterns, trends, and insights.

## Learning Outcomes Addressed
- **LO1**: Apply core principles of statistics and probability
- **LO2**: Demonstrate practical data manipulation skills with Python
- **LO3**: Analyse real-world problems using data analytics
- **LO4**: Utilise Jupyter Notebooks enhanced by AI assistance
- **LO8**: Communicate complex insights effectively

## Dataset
- **Source**: `data/cleaned/food_prices_cleaned.csv`
- **Records**: 4,798 observations
- **Time Period**: January 2007 - October 2023

---
## 1. Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Statistical libraries
from scipy import stats

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

---
## 2. Load Cleaned Data

In [ ]:
# Load the cleaned dataset
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nDate Range: {df['date'].min()} to {df['date'].max()}")
print(f"Countries: {df['country'].nunique()}")

In [ ]:
# Preview the data
df.head(10)

In [ ]:
# Data types
df.dtypes

---
## 3. Descriptive Statistics

### 3.1 Overall Summary Statistics

In [ ]:
# Statistical summary of numerical columns
numerical_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change', 'price_change_pct']
df[numerical_cols].describe()

In [ ]:
# Additional statistics: skewness and kurtosis
stats_df = pd.DataFrame({
    'Count': df[numerical_cols].count(),
    'Mean': df[numerical_cols].mean(),
    'Median': df[numerical_cols].median(),
    'Std Dev': df[numerical_cols].std(),
    'Skewness': df[numerical_cols].skew(),
    'Kurtosis': df[numerical_cols].kurtosis(),
    'Min': df[numerical_cols].min(),
    'Max': df[numerical_cols].max()
})

print("Extended Statistical Summary:")
stats_df.round(3)

### 3.2 Statistics by Country

In [ ]:
# Summary statistics by country
country_stats = df.groupby('country').agg({
    'close': ['mean', 'std', 'min', 'max'],
    'inflation': ['mean', 'std', 'min', 'max'],
    'price_range': ['mean', 'max'],
    'date': 'count'
}).round(3)

# Flatten column names
country_stats.columns = ['_'.join(col).strip() for col in country_stats.columns.values]
country_stats = country_stats.rename(columns={'date_count': 'observations'})

print("Statistics by Country:")
country_stats.sort_values('inflation_mean', ascending=False)

### 3.3 Statistics by Year

In [ ]:
# Yearly trends
yearly_stats = df.groupby('year').agg({
    'close': 'mean',
    'inflation': 'mean',
    'price_range': 'mean',
    'country': 'nunique'
}).round(3)

yearly_stats.columns = ['avg_close', 'avg_inflation', 'avg_price_range', 'countries_count']
yearly_stats

---
## 4. Data Visualisations

### 4.1 Distribution Analysis

In [ ]:
# Distribution of key variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Close price distribution
axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('Distribution of Close Prices', fontsize=12)
axes[0, 0].set_xlabel('Close Price')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['close'].mean(), color='red', linestyle='--', label=f'Mean: {df["close"].mean():.2f}')
axes[0, 0].axvline(df['close'].median(), color='green', linestyle='--', label=f'Median: {df["close"].median():.2f}')
axes[0, 0].legend()

# Inflation distribution
inflation_data = df['inflation'].dropna()
axes[0, 1].hist(inflation_data, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of Inflation Rates', fontsize=12)
axes[0, 1].set_xlabel('Inflation (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(inflation_data.mean(), color='red', linestyle='--', label=f'Mean: {inflation_data.mean():.2f}%')
axes[0, 1].legend()

# Price range distribution
axes[1, 0].hist(df['price_range'], bins=50, edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1, 0].set_title('Distribution of Price Range (Volatility)', fontsize=12)
axes[1, 0].set_xlabel('Price Range (High - Low)')
axes[1, 0].set_ylabel('Frequency')

# Box plot of inflation by year
df_with_inflation = df[df['inflation'].notna()]
sns.boxplot(data=df_with_inflation, x='year', y='inflation', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Inflation Distribution by Year', fontsize=12)
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Inflation (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/distribution_analysis.png")

### 4.2 Time Series Analysis

In [ ]:
# Global average price over time
monthly_avg = df.groupby('date').agg({
    'close': 'mean',
    'inflation': 'mean',
    'price_range': 'mean'
}).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Average close price
axes[0].plot(monthly_avg['date'], monthly_avg['close'], color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly_avg['date'], monthly_avg['close'], alpha=0.3)
axes[0].set_title('Global Average Food Price Index Over Time', fontsize=14)
axes[0].set_ylabel('Average Close Price')
axes[0].grid(True, alpha=0.3)

# Average inflation
axes[1].plot(monthly_avg['date'], monthly_avg['inflation'], color='coral', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
axes[1].fill_between(monthly_avg['date'], monthly_avg['inflation'], alpha=0.3, color='coral')
axes[1].set_title('Global Average Inflation Rate Over Time', fontsize=14)
axes[1].set_ylabel('Average Inflation (%)')
axes[1].grid(True, alpha=0.3)

# Price volatility (range)
axes[2].plot(monthly_avg['date'], monthly_avg['price_range'], color='mediumpurple', linewidth=1.5)
axes[2].set_title('Global Average Price Volatility Over Time', fontsize=14)
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Average Price Range')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/time_series_analysis.png")

### 4.3 Country Comparison

In [ ]:
# Average inflation by country
country_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=True).dropna()

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['coral' if x > 0 else 'steelblue' for x in country_inflation]
country_inflation.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_title('Average Inflation Rate by Country (2007-2023)', fontsize=14)
ax.set_xlabel('Average Inflation (%)')
ax.set_ylabel('Country')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/country_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/country_comparison.png")

In [ ]:
# Interactive line chart: Price trends by country
fig = px.line(
    df, 
    x='date', 
    y='close', 
    color='country',
    title='Food Price Index by Country Over Time',
    labels={'close': 'Price Index', 'date': 'Date', 'country': 'Country'}
)

fig.update_layout(
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5),
    hovermode='x unified'
)

fig.show()

### 4.4 Correlation Analysis

In [ ]:
# Correlation matrix
correlation_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change_pct']
correlation_matrix = df[correlation_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Matrix of Price Variables', fontsize=14)

plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/correlation_matrix.png")

In [ ]:
# Scatter plot: Price volatility vs Inflation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price range vs Inflation
scatter_data = df[df['inflation'].notna()]
axes[0].scatter(scatter_data['price_range'], scatter_data['inflation'], alpha=0.5, s=10, c='steelblue')
axes[0].set_xlabel('Price Range (Volatility)')
axes[0].set_ylabel('Inflation (%)')
axes[0].set_title('Price Volatility vs Inflation')

# Add trend line
z = np.polyfit(scatter_data['price_range'], scatter_data['inflation'], 1)
p = np.poly1d(z)
x_line = np.linspace(scatter_data['price_range'].min(), scatter_data['price_range'].max(), 100)
axes[0].plot(x_line, p(x_line), "r--", alpha=0.8, label=f'Trend line')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Close price vs Inflation
axes[1].scatter(scatter_data['close'], scatter_data['inflation'], alpha=0.5, s=10, c='coral')
axes[1].set_xlabel('Close Price')
axes[1].set_ylabel('Inflation (%)')
axes[1].set_title('Close Price vs Inflation')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/scatter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/scatter_analysis.png")

### 4.5 Seasonal Analysis

In [ ]:
# Average metrics by month
monthly_seasonal = df.groupby('month').agg({
    'close': 'mean',
    'inflation': 'mean',
    'price_range': 'mean'
}).round(3)

# Add month names
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_seasonal.index = month_names

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Average close by month
monthly_seasonal['close'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Average Close Price by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Close')
axes[0].tick_params(axis='x', rotation=45)

# Average inflation by month
monthly_seasonal['inflation'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Average Inflation by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Inflation (%)')
axes[1].tick_params(axis='x', rotation=45)

# Average volatility by month
monthly_seasonal['price_range'].plot(kind='bar', ax=axes[2], color='mediumpurple', edgecolor='black')
axes[2].set_title('Average Price Volatility by Month')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Average Price Range')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/seasonal_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/seasonal_analysis.png")
print("\nSeasonal Summary:")
monthly_seasonal

---
## 5. Key Insights Summary

### Insights from Exploratory Data Analysis

In [ ]:
# Generate summary insights
print("="*60)
print("KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("="*60)

# 1. Overall statistics
print("\n1. OVERALL STATISTICS")
print(f"   - Average close price: {df['close'].mean():.3f}")
print(f"   - Average inflation: {df['inflation'].mean():.2f}%")
print(f"   - Price range: {df['close'].min():.3f} to {df['close'].max():.3f}")
print(f"   - Inflation range: {df['inflation'].min():.2f}% to {df['inflation'].max():.2f}%")

# 2. Top/Bottom countries by inflation
print("\n2. COUNTRIES WITH HIGHEST AVERAGE INFLATION")
top_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=False).head(5)
for country, inflation in top_inflation.items():
    print(f"   - {country}: {inflation:.2f}%")

print("\n3. COUNTRIES WITH LOWEST AVERAGE INFLATION")
bottom_inflation = df.groupby('country')['inflation'].mean().sort_values().head(5)
for country, inflation in bottom_inflation.items():
    print(f"   - {country}: {inflation:.2f}%")

# 3. Correlation insights
print("\n4. KEY CORRELATIONS")
corr_price_inflation = df['close'].corr(df['inflation'])
corr_volatility_inflation = df['price_range'].corr(df['inflation'])
print(f"   - Price vs Inflation correlation: {corr_price_inflation:.3f}")
print(f"   - Volatility vs Inflation correlation: {corr_volatility_inflation:.3f}")

# 4. Time trends
print("\n5. TRENDS OVER TIME")
first_year_avg = df[df['year'] == df['year'].min()]['close'].mean()
last_year_avg = df[df['year'] == df['year'].max()]['close'].mean()
price_change_total = ((last_year_avg - first_year_avg) / first_year_avg) * 100
print(f"   - Price index change from {df['year'].min()} to {df['year'].max()}: {price_change_total:.1f}%")

# 5. Volatility insights
print("\n6. VOLATILITY ANALYSIS")
most_volatile = df.groupby('country')['price_range'].mean().sort_values(ascending=False).head(3)
print("   Most volatile countries (by avg price range):")
for country, vol in most_volatile.items():
    print(f"   - {country}: {vol:.3f}")

print("\n" + "="*60)

---
## 6. Next Steps

Based on this exploratory analysis, the next steps include:

1. **Hypothesis Testing** (see `Hypothesis_Testing.ipynb`)
   - Test if inflation differs significantly between regions
   - Test correlation significance between price volatility and inflation
   - Test for seasonal patterns in food prices

2. **Predictive Modelling**
   - Time series forecasting for price trends
   - Regression analysis for inflation prediction

3. **Dashboard Development**
   - Create interactive Power BI/Tableau dashboard
   - Enable filtering by country, date range, and metrics

---
## AI Assistance Documentation

This notebook was developed with AI assistance (GitHub Copilot) for:
- Code generation and optimization
- Visualization best practices
- Statistical analysis guidance
- Documentation and comments

All AI-generated code was reviewed, tested, and validated by the analyst.